In [ ]:
!pip install librosa -q

import os, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tv_models
import librosa
warnings.filterwarnings('ignore')

print(f'torch: {torch.__version__}')

# Audio config — must match training
SR = 32000; DURATION = 5; N_FFT = 1024; HOP_LEN = 320
N_MELS = 128; FMIN = 20; FMAX = 16000; IMG_W = 160
INFER_BATCH = 64

In [ ]:
# ── Find competition data ─────────────────────────────────────────────
COMP_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = [d for d in dirs if d not in ('train_audio', 'test_soundscapes', 'train_soundscapes')]
    if 'sample_submission.csv' in files:
        COMP_DIR = root
        break
assert COMP_DIR, 'sample_submission.csv not found'
print(f'COMP_DIR: {COMP_DIR}')

# ── Find all trained model checkpoints ───────────────────────────────
# kernel_sources mount at /kaggle/input/notebooks/{username}/{kernel-slug}/
BASE = '/kaggle/input/notebooks/gorubachohu'
MODEL_DIRS = [
    f'{BASE}/birdclef2026-efficientnet-training',   # fold 0
    f'{BASE}/birdclef2026-training-fold1',           # fold 1
    f'{BASE}/birdclef2026-training-fold2',           # fold 2
]
MODEL_PATHS = []
for d in MODEL_DIRS:
    if os.path.exists(d):
        for f in sorted(os.listdir(d)):
            if f.endswith('.pt'):
                MODEL_PATHS.append(os.path.join(d, f))

print(f'Found {len(MODEL_PATHS)} model(s):')
for p in MODEL_PATHS:
    print(f'  {p}')

In [ ]:
# ── Model definition (must match training) ────────────────────────────
class BirdModel(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        base = tv_models.efficientnet_b0(weights=None)
        self.features = base.features
        self.pool     = base.avgpool
        n_feat        = base.classifier[1].in_features
        self.head     = nn.Sequential(nn.Dropout(0.3), nn.Linear(n_feat, n_classes))

    def forward(self, x):
        return self.head(torch.flatten(self.pool(self.features(x)), 1))

# Load first checkpoint to get species list
# weights_only=False needed because checkpoint contains species list (Python strings)
ckpt0 = torch.load(MODEL_PATHS[0], map_location='cpu', weights_only=False)
species_list = ckpt0['species']
N_CLASSES = len(species_list)
print(f'Species: {N_CLASSES}')

# Load all models
models = []
for path in MODEL_PATHS:
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    m = BirdModel(N_CLASSES)
    m.load_state_dict(ckpt['state'])
    m.eval()
    models.append(m)
    print(f'  Loaded {os.path.basename(path)} (val_auc={ckpt["auc"]:.4f})')

print(f'Ensemble size: {len(models)}')

In [ ]:
# ── Audio → image ─────────────────────────────────────────────────────
def audio_to_img(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP_LEN,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    mel = mel.astype(np.float32)
    if mel.shape[1] != IMG_W:
        mel = np.array([
            np.interp(np.linspace(0, mel.shape[1]-1, IMG_W),
                      np.arange(mel.shape[1]), mel[i])
            for i in range(mel.shape[0])], dtype=np.float32)
    return np.stack([mel, mel, mel], axis=0)

In [ ]:
# ── Load submission template & parse row_ids ──────────────────────────
sub_df = pd.read_csv(f'{COMP_DIR}/sample_submission.csv')
species_cols = [c for c in sub_df.columns if c != 'row_id']
print(f'Submission rows: {len(sub_df):,} | species: {len(species_cols)}')

# row_id format: {soundscape_name}_{end_second}
# e.g. XC123456_5, XC123456_10, ...
from collections import defaultdict
soundscape_chunks = defaultdict(list)
for row_id in sub_df['row_id']:
    sc_name, end_sec = row_id.rsplit('_', 1)
    soundscape_chunks[sc_name].append(int(end_sec))

print(f'Soundscapes: {len(soundscape_chunks)}')

# ── Run inference ─────────────────────────────────────────────────────
SOUNDSCAPE_DIR = f'{COMP_DIR}/test_soundscapes'
predictions = {}   # row_id -> np.ndarray (N_CLASSES,)

t_start = time.time()
for sc_idx, (sc_name, end_secs) in enumerate(soundscape_chunks.items()):
    sc_path = f'{SOUNDSCAPE_DIR}/{sc_name}.ogg'

    try:
        audio, _ = librosa.load(sc_path, sr=SR, mono=True)
    except Exception as e:
        print(f'[WARN] {sc_name}: {e}')
        audio = np.zeros(SR * 60, dtype=np.float32)

    # Build batch of images for all chunks in this soundscape
    imgs, row_ids = [], []
    for end_sec in sorted(end_secs):
        start_s = max(0, int((end_sec - DURATION) * SR))
        end_s   = int(end_sec * SR)
        chunk   = audio[start_s:end_s]
        if len(chunk) < DURATION * SR:
            chunk = np.pad(chunk, (0, DURATION * SR - len(chunk)))
        imgs.append(audio_to_img(chunk))
        row_ids.append(f'{sc_name}_{end_sec}')

    imgs_arr = np.stack(imgs)   # (T, 3, H, W)

    # Ensemble inference in batches
    all_probs = np.zeros((len(imgs_arr), N_CLASSES), dtype=np.float32)
    for b_start in range(0, len(imgs_arr), INFER_BATCH):
        batch = torch.from_numpy(imgs_arr[b_start:b_start+INFER_BATCH])
        batch_probs = np.zeros((len(batch), N_CLASSES), dtype=np.float32)
        for model in models:
            with torch.no_grad():
                batch_probs += torch.sigmoid(model(batch)).numpy()
        batch_probs /= len(models)
        all_probs[b_start:b_start+len(batch)] = batch_probs

    for i, row_id in enumerate(row_ids):
        predictions[row_id] = all_probs[i]

    if (sc_idx + 1) % 10 == 0 or sc_idx == 0:
        elapsed = time.time() - t_start
        print(f'  [{sc_idx+1}/{len(soundscape_chunks)}] {elapsed:.1f}s elapsed')

print(f'Inference complete in {time.time()-t_start:.1f}s')

In [ ]:
# ── Build submission CSV ───────────────────────────────────────────────
pred_matrix = np.zeros((len(sub_df), len(species_cols)), dtype=np.float32)
for i, row_id in enumerate(sub_df['row_id']):
    if row_id in predictions:
        pred_matrix[i] = predictions[row_id]

sub_df[species_cols] = pred_matrix
sub_df.to_csv('/kaggle/working/submission.csv', index=False)

print(f'Saved submission.csv  shape={sub_df.shape}')
print(f'Coverage: {sum(r in predictions for r in sub_df["row_id"])}/{len(sub_df)} rows')
print(sub_df.head(3))